In [ ]:
import sys
sys.path.append('../src/')

from dataset import gee_data as gd
from dataset import feature_extraction as fe
from dataset import glamos_processing as glamos
import ee

In [ ]:
gdf = glamos.get_data(2000, 2025)

In [ ]:
gd.initialize_gee('ee-jaybrandon-dspro2')

In [ ]:
def assign_satellite_label(date):
    year = date.year
    if year < 1984:
        return None
    elif 1984 <= year < 2013:
        return "landsat5"
    elif 2013 <= year <= 2016:
        return "landsat8"
    else:
        return "sentinel2"

In [ ]:
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(epsg=4326)
gdf["geometry"] = gdf.geometry.buffer(0)
gdf["satellite"] = gdf["observation_end"].apply(assign_satellite_label)

In [ ]:
gdf

In [ ]:
row = gdf.iloc[0]

In [ ]:
row

In [ ]:
roi = ee.Geometry(row.geometry.__geo_interface__)

collection = gd.get_glacier_collection(
    sensor_type=row['satellite'],
    polygon=roi,
    start_date=row["observation_start"].strftime("%Y-%m-%d"),
    end_date=row["observation_end"].strftime("%Y-%m-%d"),
    cloud_threshold=40,
)

In [ ]:
collection

In [ ]:
def add_date(img):
        date_str = img.date().format('YYYY-MM-DD')
        return img.set('date', date_str)
    
col_with_date = collection.map(add_date)

In [ ]:
l1 = col_with_date.aggregate_histogram('date').keys()

In [ ]:
l2 = col_with_date.aggregate_array('date').distinct()

In [ ]:
col_with_date.propertyNames().getInfo()

In [ ]:
l2.getInfo()

In [ ]:
l1.getInfo()

In [ ]:
dem = gd.get_dem(roi)

In [ ]:
results = fe.extract_glacier_period_features(collection, dem, roi, row['obs_id'])

In [ ]:
results

In [ ]:
results.get("final_mask_image")

In [ ]:
results.get("features")[0]